# レッスン 11 - モデルコンテキストプロトコル (MCP)

**モデルコンテキストプロトコル (MCP)** は、エージェントが実行時に動的にツール、リソース、データソースを発見して使用できるオープン標準です。ツールをエージェントにハードコーディングする代わりに、MCPはエージェントがオンデマンドで機能を公開する外部サーバーに接続できるようにします。

このレッスンでは、次の内容を学びます:
- MCPとは何か、そしてそれがエージェントシステムにとって重要な理由
- MCPのクライアントサーバーアーキテクチャの仕組み
- MCPスタイルのツール検出を使うエージェントの作り方


## セットアップ

**前提条件:**
- デプロイされたモデルを持つMicrosoft Foundryプロジェクト
- 認証のために`az login`を実行する


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## モデルコンテキストプロトコル（MCP）とは？

MCPは、AIエージェントが外部のツールやデータソースを発見し、連携するための標準的な方法を定義しています：

- **MCPサーバー**：標準的なプロトコルを通じてツール、リソース、プロンプトを公開します
- **MCPクライアント**：サーバーに接続し利用可能な機能を発見するエージェントランタイム
- <strong>動的発見</strong>：エージェントはツールをハードコードする必要がなく、実行時に利用可能なものを発見します

これにより、エージェントのコードを変更せずに新しい機能を追加できる拡張性の高いエージェントシステムを構築することが可能です。


## MCPの仕組み

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. エージェント（MCPクライアント）がMCPサーバーに接続します
2. サーバーは利用可能なツールとそのスキーマのリストで応答します
3. エージェントは推論中に検出された任意のツールを呼び出すことができます
4. 結果は同じプロトコルを通じて戻されます


## MCPツール検出のシミュレーション

実際のMCPサーバーには稼働中のサーバープロセスが必要なため、MCPに接続された宿泊サービスが提供するものをシミュレートする `@tool` 関数を使ってパターンを示します。

本番環境では、これらのツールはローカルに定義されるのではなく、MCPサーバーから動的に検出されます。


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCPスタイルツールでエージェントを構築する


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## 本番環境における MCP

本番環境では、MCP は強力なパターンを可能にします：

- <strong>動的ツール検出</strong>：エージェントは MCP サーバーに接続し、実行時にツールを検出します
- <strong>分離されたアーキテクチャ</strong>：ツールプロバイダーはエージェントとは独立して更新が可能です
- <strong>組織横断的共有</strong>：チームは MCP サーバーを通じて機能を公開し、どのエージェントも利用できます
- **Microsoft Agent Framework サポート**：MAF は `mcp` 統合を介した組み込みの MCP クライアントサポートを備えています

MAF で本物の MCP サーバーを使うには、`hosted_mcp_tool()` または MCP クライアント統合を通じて接続します。

**詳細はこちら：**
- [MCP Specification](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP Support](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## まとめ

このレッスンで学んだこと：
- **MCP** はエージェントとツールプロバイダー間の動的ツール発見のためのオープン標準です
- <strong>クライアントサーバーアーキテクチャ</strong> により、エージェントは実行時に機能を発見できます
- MCPは<strong>拡張可能で分離されたエージェントシステム</strong>を可能にし、ツールをコード変更なしで追加できます
- Microsoft Agent Frameworkは<strong>本番利用のための組み込みMCPサポート</strong>を提供しています


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
